In [2]:
import os, pickle
import pandas as pd, numpy as np
os.chdir('/Users/Jake/Documents/Cricket Comps Data')

pool = pd.read_pickle('pool.pkl')
base = pd.read_pickle('base.pkl')
dist = pd.read_pickle('dist.pkl')
bat  = pd.read_pickle('bat_factors.pkl')
bowl = pd.read_pickle('bowl_factors.pkl')
ven  = pd.read_pickle('venue_factors.pkl')
with open('shrinkage_K.pkl','rb') as f: Ks = pickle.load(f)
with open('K_venue.pkl','rb') as f: K_ven = pickle.load(f)

legal = pool[pool['legal']].copy()
outcomes = np.array([0,1,2,3,4,5,6])
_pcache = {}
print(len(pool), 'balls loaded')

404664 balls loaded


In [3]:
def phase(over):
    if over <= 5:   return 'powerplay'
    if over <= 14:  return 'middle'
    return 'death'


def get_factor(df, name, ph, metric, role):
    K = Ks[(role, metric, ph)]
    col = 'runs_factor' if metric == 'runs' else 'wkt_factor'
    try:
        row = df.loc[(name, ph)]
    except KeyError:
        return 1.0
    return (row['balls'] * row[col] + K) / (row['balls'] + K)


def venue_factor(v, ph):
    K = K_ven[('runs', ph)]
    try:
        row = ven.loc[(v, ph)]
    except KeyError:
        return 1.0
    return (row['balls'] * row['runs_factor'] + K) / (row['balls'] + K)


def build_lookup(team, attack, venue):
    L = {}
    for ph in ['powerplay','middle','death']:
        for p in team:
            L[('bat', p, ph)] = (get_factor(bat, p, ph, 'runs', 'batter'),
                                 get_factor(bat, p, ph, 'wicket', 'batter'))
        for p in attack:
            L[('bowl', p, ph)] = (get_factor(bowl, p, ph, 'runs', 'bowler'),
                                  get_factor(bowl, p, ph, 'wicket', 'bowler'))
        L[('ven', ph)] = venue_factor(venue, ph)
    return L


def ball_probs(batter, bowler, ph, L):
    key = (batter, bowler, ph)
    if key in _pcache:
        return _pcache[key]

    p = dist.loc[ph].values.astype(float).copy()
    br, bw = L[('bat', batter, ph)]
    or_, ow = L[('bowl', bowler, ph)]
    vfac = L[('ven', ph)]

    p_wkt = min(max(base.loc[ph,'wkt'] * bw * ow, 0.005), 0.35)
    target = base.loc[ph,'rpb'] * br * or_ * vfac

    outs = outcomes.astype(float)
    for _ in range(40):
        s = target / max((p*outs).sum() * (1-p_wkt), 1e-9)
        p[1:] = np.minimum(p[1:] * s, 1.0)
        tot = p[1:].sum()
        if tot > 0.98:
            p[1:] *= 0.98/tot
            tot = 0.98
        p[0] = 1 - tot

    res = (np.cumsum(p * (1-p_wkt)), p_wkt)
    _pcache[key] = res
    return res


def sim_innings(bat_order, bowl_attack, L, rng, target=None):
    score, wkts = 0, 0
    strike, nonstrike, next_bat = 0, 1, 2

    for over in range(20):
        if wkts >= 10: break
        ph = phase(over)
        bowler = bowl_attack[over % len(bowl_attack)]

        for _ in range(6):
            if wkts >= 10: break
            if target is not None and score >= target: break

            cum, p_wkt = ball_probs(bat_order[strike], bowler, ph, L)
            r = rng.random()
            if r < p_wkt:
                wkts += 1
                if next_bat < len(bat_order):
                    strike = next_bat; next_bat += 1
            else:
                runs = int(outcomes[np.searchsorted(cum, r - p_wkt)])
                score += runs
                if runs % 2 == 1:
                    strike, nonstrike = nonstrike, strike

        strike, nonstrike = nonstrike, strike
        if target is not None and score >= target: break

    return score, wkts


def price_match(teamA, attackA, teamB, attackB, venue, n=10000, seed=1):
    L = build_lookup(teamA + teamB, attackA + attackB, venue)
    rng = np.random.default_rng(seed)
    a_wins = 0.0
    for i in range(n):
        a_first = (i % 2 == 0)
        if a_first:
            s1, _ = sim_innings(teamA, attackB, L, rng)
            s2, _ = sim_innings(teamB, attackA, L, rng, target=s1+1)
        else:
            s1, _ = sim_innings(teamB, attackA, L, rng)
            s2, _ = sim_innings(teamA, attackB, L, rng, target=s1+1)
        if s2 == s1:
            a_wins += 0.5
        elif (s2 > s1) != a_first:
            a_wins += 1
    p = a_wins / n
    return p, 1/p, 1/(1-p)


barbados = ['BA King','ZN Carter','Q de Kock','SE Rutherford','C Green',
            'RA Clarke','KA Anderson','G Motie','DR Sams',
            'Mujeeb Ur Rahman','RR Simmonds']
patriots = ['J Charles','KR Mayers','MD Shanaka','JO Holder','ADS Fletcher',
            'PWH de Silva','A Athanaze','N Bidaisee','Naseem Shah',
            'JS Louis','Waqar Salamkheil']
barb_attack = ['DR Sams','RR Simmonds','Mujeeb Ur Rahman','G Motie','SE Rutherford']
pat_attack  = ['Naseem Shah','JO Holder','PWH de Silva','JS Louis','KR Mayers']

print('functions loaded')

functions loaded


In [3]:
# rebuild a scorecard from the raw data
one = balls[balls['match'] == balls['match'].iloc[0]]
print(one.groupby('innings')['runs_total'].sum())
print()
print(one.groupby('innings')['wicket'].sum())
print()
print(one[['comp','date','venue']].iloc[0])
print()
print(balls['date'].min(), '->', balls['date'].max())

NameError: name 'balls' is not defined

In [ ]:
# rebuild a scorecard from the raw data
one = balls[balls['match'] == balls['match'].iloc[0]]
print(one.groupby('innings')['runs_total'].sum())
print()
print(one.groupby('innings')['wicket'].sum())
print()
print(one[['comp','date','venue']].iloc[0])
print()
print(balls['date'].min(), '->', balls['date'].max())

In [ ]:
def phase(over):
    if over <= 5:   return 'powerplay'
    if over <= 14:  return 'middle'
    return 'death'

pool['phase'] = pool['over'].apply(phase)

base = (pool.groupby('phase')
        .agg(balls=('runs_total','size'),
             rpb=('runs_total','mean'),
             wkt=('wicket','mean'))
        .round(4))
print(base)

In [ ]:
pool['legal'] = ~pool['extras'].apply(lambda e: 'wides' in e or 'noballs' in e)

base = (pool.groupby('phase')
        .agg(deliveries=('runs_total','size'),
             legal_balls=('legal','sum'),
             runs=('runs_total','sum'),
             wickets=('wicket','sum')))

base['rpb'] = (base['runs'] / base['legal_balls']).round(4)
base['wkt'] = (base['wickets'] / base['legal_balls']).round(4)
print(base[['legal_balls','rpb','wkt']])

In [ ]:
d = pool[pool['phase']=='death']
inn = d.groupby(['match','innings']).agg(
        balls=('legal','sum'), runs=('runs_total','sum'))
print(inn['balls'].describe())
print()
print('full 30-ball death overs:', (inn['balls']>=30).sum(), 'of', len(inn))
print()
full = inn[inn['balls']>=30]
print('rpb when death fully bowled:', round(full['runs'].sum()/full['balls'].sum(), 3))

In [ ]:
d = pool[pool['phase']=='death']
by_comp = d.groupby('comp').apply(
    lambda g: pd.Series({
        'legal': g['legal'].sum(),
        'rpb': round(g['runs_total'].sum()/g['legal'].sum(), 3)
    }))
print(by_comp)

In [ ]:
ipl = pool[(pool['comp']=='ipl') & (pool['phase']=='death')].copy()
ipl['yr'] = ipl['date'].dt.year
print(ipl.groupby('yr').apply(
    lambda g: pd.Series({'legal': g['legal'].sum(),
                         'rpb': round(g['runs_total'].sum()/g['legal'].sum(),3)})))

In [ ]:
pool = balls[balls['date'] >= '2022-01-01'].copy()
pool['legal'] = ~pool['extras'].apply(lambda e: 'wides' in e or 'noballs' in e)
pool['phase'] = pool['over'].apply(phase)
print(len(pool), 'balls')
print(pool.groupby('comp').size())

In [ ]:
import numpy as np

latest = pool['date'].max()
age_years = (latest - pool['date']).dt.days / 365.25
pool['w'] = 0.5 ** (age_years / 2.0)

print('newest weight:', round(pool['w'].max(), 3))
print('oldest weight:', round(pool['w'].min(), 3))

In [ ]:
def wrate(g):
    lw = (g['legal'] * g['w']).sum()
    return pd.Series({
        'eff_balls': round(lw),
        'rpb': round((g['runs_total'] * g['w']).sum() / lw, 4),
        'wkt': round((g['wicket'] * g['w']).sum() / lw, 4),
    })

base = pool.groupby('phase').apply(wrate)
print(base)

In [ ]:
legal = pool[pool['legal']]
dist = (legal.groupby(['phase','runs_bat'])['w'].sum()
        .unstack(fill_value=0))
dist = dist.div(dist.sum(axis=1), axis=0).round(4)
print(dist)

In [ ]:
pool.to_pickle('pool.pkl')
base.to_pickle('base.pkl')
dist.to_pickle('dist.pkl')
print('saved')

In [ ]:
import os, glob, json
import pandas as pd, numpy as np
os.chdir('/Users/Jake/Documents/Cricket Comps Data')

pool = pd.read_pickle('pool.pkl')
base = pd.read_pickle('base.pkl')
dist = pd.read_pickle('dist.pkl')

print(len(pool), 'balls')
print(base)

In [ ]:
legal = pool[pool['legal']].copy()

bat = (legal.groupby(['batter','phase'])
       .apply(lambda g: pd.Series({
           'balls': g['legal'].sum(),
           'eff_balls': (g['w']).sum(),
           'runs_w': (g['runs_bat'] * g['w']).sum(),
           'wkts_w': (g['wicket'] * g['w']).sum(),
       })))

bat['rpb'] = bat['runs_w'] / bat['eff_balls']
bat['wpb'] = bat['wkts_w'] / bat['eff_balls']

bat = bat.join(base[['rpb','wkt']].rename(
        columns={'rpb':'base_rpb','wkt':'base_wkt'}), on='phase')

bat['runs_factor'] = (bat['rpb'] / bat['base_rpb']).round(3)
bat['wkt_factor']  = (bat['wpb'] / bat['base_wkt']).round(3)

print(len(bat), 'player-phase rows')

In [ ]:
death = bat.xs('death', level='phase')
death = death[death['balls'] >= 300].sort_values('runs_factor', ascending=False)
print(death[['balls','runs_factor','wkt_factor']].head(20))

In [ ]:
bowl = (legal.groupby(['bowler','phase'])
        .apply(lambda g: pd.Series({
            'balls': g['legal'].sum(),
            'eff_balls': (g['w']).sum(),
            'runs_w': (g['runs_total'] * g['w']).sum(),
            'wkts_w': (g['wicket'] * g['w']).sum(),
        })))

bowl['rpb'] = bowl['runs_w'] / bowl['eff_balls']
bowl['wpb'] = bowl['wkts_w'] / bowl['eff_balls']

bowl = bowl.join(base[['rpb','wkt']].rename(
         columns={'rpb':'base_rpb','wkt':'base_wkt'}), on='phase')

bowl['runs_factor'] = (bowl['rpb'] / bowl['base_rpb']).round(3)
bowl['wkt_factor']  = (bowl['wpb'] / bowl['base_wkt']).round(3)

print(len(bowl), 'bowler-phase rows')

In [ ]:
d = bowl.xs('death', level='phase')
d = d[d['balls'] >= 300].sort_values('runs_factor')
print(d[['balls','runs_factor','wkt_factor']].head(15))

In [ ]:
b = bat.xs('death', level='phase')
for lo, hi in [(50,100),(100,200),(200,400),(400,800),(800,5000)]:
    s = b[(b['balls']>=lo) & (b['balls']<hi)]['runs_factor']
    print(f'{lo}-{hi}: n={len(s)}, sd={s.std():.3f}')

In [ ]:
d = legal[legal['phase']=='death'].copy().sort_values('date')

# split each player's balls in half by time
d['seq'] = d.groupby('batter').cumcount()
d['n'] = d.groupby('batter')['batter'].transform('size')
first = d[d['seq'] < d['n']/2]
second = d[d['seq'] >= d['n']/2]

base_rpb = base.loc['death','rpb']

f = first.groupby('batter').agg(balls=('legal','sum'), runs=('runs_bat','sum'))
s = second.groupby('batter').agg(balls=('legal','sum'), runs=('runs_bat','sum'))

f['raw'] = (f['runs']/f['balls']) / base_rpb
s['actual'] = (s['runs']/s['balls']) / base_rpb

j = f.join(s[['balls','actual']], rsuffix='_2', how='inner')
j = j[(j['balls']>=30) & (j['balls_2']>=30)]
print(len(j), 'players')

for K in [0, 50, 100, 150, 200, 300, 400, 600, 900, 1500, 99999]:
    pred = (j['balls']*j['raw'] + K) / (j['balls'] + K)
    err = ((pred - j['actual'])**2 * j['balls_2']).sum() / j['balls_2'].sum()
    print(f'K={K:>6}  error={err:.5f}')

In [ ]:
def find_K(role, metric, ph, min_balls=30):
    d = legal[legal['phase']==ph].copy().sort_values('date')
    col = 'runs_bat' if (role=='batter' and metric=='runs') else \
          'runs_total' if metric=='runs' else 'wicket'
    baseline = base.loc[ph, 'rpb' if metric=='runs' else 'wkt']

    d['seq'] = d.groupby(role).cumcount()
    d['n']   = d.groupby(role)[role].transform('size')
    f = d[d['seq'] <  d['n']/2].groupby(role).agg(balls=('legal','sum'), v=(col,'sum'))
    s = d[d['seq'] >= d['n']/2].groupby(role).agg(balls=('legal','sum'), v=(col,'sum'))

    f['raw']    = (f['v']/f['balls']) / baseline
    s['actual'] = (s['v']/s['balls']) / baseline
    j = f.join(s[['balls','actual']], rsuffix='_2', how='inner')
    j = j[(j['balls']>=min_balls) & (j['balls_2']>=min_balls)]

    best = None
    for K in [25,50,75,100,150,200,300,400,600,900,1500,2500,4000]:
        pred = (j['balls']*j['raw'] + K) / (j['balls'] + K)
        err  = ((pred - j['actual'])**2 * j['balls_2']).sum() / j['balls_2'].sum()
        if best is None or err < best[1]:
            best = (K, err)
    return best[0], round(best[1],5), len(j)

Ks = {}
for role in ['batter','bowler']:
    for metric in ['runs','wicket']:
        for ph in ['powerplay','middle','death']:
            K, err, n = find_K(role, metric, ph)
            Ks[(role,metric,ph)] = K
            print(f'{role:<7} {metric:<7} {ph:<10} K={K:<5} err={err} n={n}')

In [ ]:
import pickle
with open('shrinkage_K.pkl','wb') as f:
    pickle.dump(Ks, f)

bat.to_pickle('bat_factors.pkl')
bowl.to_pickle('bowl_factors.pkl')
print('saved')

## Shrinkage constants — method and findings

**Problem.** Raw player factors are unreliable on small samples. A batter
who strikes at 1.6x the league rate off 60 balls is probably lucky, not
elite. Shrinkage pulls unreliable factors back toward 1.0 (league average),
in proportion to how little data supports them:

    used_factor = (balls x raw_factor + K) / (balls + K)

K is measured in balls: the sample size at which a player's own record and
the league average carry equal weight.

**Method.** Split each player's balls in half chronologically. Compute raw
factors from the first half, shrink with a candidate K, then measure squared
error against their actual second-half rate. Errors weighted by second-half
balls. Repeat across K values and take the minimum. This is out-of-sample —
K is chosen on data it wasn't fitted to.

**Initial estimate was wrong.** Eyeballing the standard deviation of factors
by sample-size band suggested K around 300. Out-of-sample testing gave 100.
The visual method conflates real skill spread with noise, so it flattens
later than the true optimum.

**Shrinkage is not optional.** For batting runs at the death, K=0 (raw
factors) scored 0.0329 — worse than K=infinity (0.0309), i.e. worse than
ignoring player data entirely. Unshrunk small-sample factors are actively
misleading.

**Chosen K values:**

| Role   | Metric | Powerplay | Middle | Death |
|--------|--------|-----------|--------|-------|
| Batter | Runs   | 100       | 100    | 75    |
| Batter | Wicket | 600       | 400    | 300   |
| Bowler | Runs   | 400       | 400    | 300   |
| Bowler | Wicket | 4000      | 2500   | 900   |

**Three findings:**

1. Runs are far more learnable than wickets. Batting runs settle at K=75-100;
   wicket factors need 300-4000. Wickets are rare events, so per-ball
   dismissal rates take huge samples to mean anything.
2. Batters are more distinguishable than bowlers on runs (K=100 vs 400) —
   the batter has more control over a ball's outcome.
3. Every K falls as the innings progresses. Bowling wickets go 4000 →
   2500 → 900. Skill separates most at the death.

Bowling wicket factors in the powerplay (K=4000) exceed almost any bowler's
sample, so in practice the model treats powerplay bowlers as near-average
for dismissals. This is a real result, not a failure of the method.

**Caveat.** K was tuned on the full pool (7 competitions, 2022+, recency-
weighted). It has not been re-tested on CPL alone.

## Open question — bowler types

Today's shrinkage test measured whether an *individual* bowler's past
wicket rate predicts his own future wicket rate. In the powerplay it
barely does (K=4000, larger than almost any bowler's sample in the pool).

This is NOT the same as saying bowler types don't matter. That question —
do left-arm seamers, or wrist-spinners, take more powerplay wickets as a
group? — pools thousands of balls per category rather than a few hundred
per player, so an effect could be clearly detectable at type level while
invisible at individual level.

Untested here. Cricsheet does not carry bowling style, so testing it needs
style data attached from another source. Worth doing: if type effects are
real, they would give the model something to use in the powerplay where
individual factors are near-useless.

Claim to stick to for now: at available sample sizes, individual bowlers'
powerplay wicket rates do not predict their own future powerplay wicket
rates. Nothing stronger.

In [ ]:
import os, glob, json, pickle
import pandas as pd, numpy as np
os.chdir('/Users/Jake/Documents/Cricket Comps Data')

pool = pd.read_pickle('pool.pkl')
base = pd.read_pickle('base.pkl')
dist = pd.read_pickle('dist.pkl')
bat  = pd.read_pickle('bat_factors.pkl')
bowl = pd.read_pickle('bowl_factors.pkl')
with open('shrinkage_K.pkl','rb') as f:
    Ks = pickle.load(f)

legal = pool[pool['legal']].copy()
print(len(pool), 'balls |', len(Ks), 'K values')
print(base)

In [ ]:
v = legal.groupby('venue')['legal'].sum().sort_values(ascending=False)
print(len(v), 'venues')
print(v.head(25))

In [ ]:
cpl_v = legal[legal['comp']=='cpl'].groupby('venue')['legal'].sum().sort_values(ascending=False)
print(cpl_v)

In [ ]:
ven = (legal.groupby(['venue','phase'])
       .apply(lambda g: pd.Series({
           'balls': g['legal'].sum(),
           'eff_balls': g['w'].sum(),
           'runs_w': (g['runs_total'] * g['w']).sum(),
           'wkts_w': (g['wicket'] * g['w']).sum(),
       })))

ven['rpb'] = ven['runs_w'] / ven['eff_balls']
ven['wpb'] = ven['wkts_w'] / ven['eff_balls']
ven = ven.join(base[['rpb','wkt']].rename(
        columns={'rpb':'base_rpb','wkt':'base_wkt'}), on='phase')
ven['runs_factor'] = (ven['rpb'] / ven['base_rpb']).round(3)
ven['wkt_factor']  = (ven['wpb'] / ven['base_wkt']).round(3)

cpl_grounds = legal[legal['comp']=='cpl']['venue'].unique()
out = ven[ven.index.get_level_values('venue').isin(cpl_grounds)]
print(out.xs('middle', level='phase')[['balls','runs_factor','wkt_factor']].round(3))

In [ ]:
comp = (legal.groupby(['comp','phase'])
        .apply(lambda g: pd.Series({
            'rpb': (g['runs_total']*g['w']).sum() / g['w'].sum(),
            'wpb': (g['wicket']*g['w']).sum() / g['w'].sum(),
        })))
comp = comp.join(base[['rpb','wkt']].rename(columns={'rpb':'b_r','wkt':'b_w'}), on='phase')
comp['comp_runs'] = (comp['rpb']/comp['b_r']).round(3)
comp['comp_wkt']  = (comp['wpb']/comp['b_w']).round(3)
print(comp[['comp_runs','comp_wkt']].unstack(level='phase'))

In [ ]:
legal2 = legal.join(comp[['comp_runs','comp_wkt']], on=['comp','phase'])

ven = (legal2.groupby(['venue','phase'])
       .apply(lambda g: pd.Series({
           'balls': g['legal'].sum(),
           'eff': g['w'].sum(),
           'exp_r': (g['comp_runs'] * g['w']).sum(),
           'exp_w': (g['comp_wkt']  * g['w']).sum(),
           'act_r': (g['runs_total']/g['runs_total'].mean() * 0 + g['runs_total'] * g['w']).sum(),
           'act_w': (g['wicket'] * g['w']).sum(),
       })))

ven = ven.join(base[['rpb','wkt']].rename(columns={'rpb':'b_r','wkt':'b_w'}), on='phase')
ven['runs_factor'] = (ven['act_r'] / (ven['exp_r'] * ven['b_r'])).round(3)
ven['wkt_factor']  = (ven['act_w'] / (ven['exp_w'] * ven['b_w'])).round(3)

out = ven[ven.index.get_level_values('venue').isin(cpl_grounds)]
print(out.xs('middle', level='phase')[['balls','runs_factor','wkt_factor']])

In [ ]:
def find_K_venue(metric, ph, min_balls=100):
    d = legal2[legal2['phase']==ph].copy().sort_values('date')
    col = 'runs_total' if metric=='runs' else 'wicket'
    expcol = 'comp_runs' if metric=='runs' else 'comp_wkt'
    b = base.loc[ph, 'rpb' if metric=='runs' else 'wkt']

    d['seq'] = d.groupby('venue').cumcount()
    d['n']   = d.groupby('venue')['venue'].transform('size')
    f = d[d['seq'] <  d['n']/2].groupby('venue').agg(balls=('legal','sum'), v=(col,'sum'), e=(expcol,'sum'))
    s = d[d['seq'] >= d['n']/2].groupby('venue').agg(balls=('legal','sum'), v=(col,'sum'), e=(expcol,'sum'))

    f['raw']    = f['v'] / (f['e']*b)
    s['actual'] = s['v'] / (s['e']*b)
    j = f.join(s[['balls','actual']], rsuffix='_2', how='inner')
    j = j[(j['balls']>=min_balls) & (j['balls_2']>=min_balls)]

    best = None
    for K in [100,250,500,1000,2000,4000,8000,16000,32000,64000,128000,10**9]:
        pred = (j['balls']*j['raw'] + K) / (j['balls'] + K)
        err  = ((pred-j['actual'])**2 * j['balls_2']).sum()/j['balls_2'].sum()
        if best is None or err < best[1]:
            best = (K, err)
    return best[0], round(best[1],5), len(j)

for metric in ['runs','wicket']:
    for ph in ['powerplay','middle','death']:
        print(metric, ph, find_K_venue(metric, ph))

In [ ]:
K_ven = {('runs','powerplay'):4000, ('runs','middle'):4000, ('runs','death'):2000}
ven.to_pickle('venue_factors.pkl')
with open('K_venue.pkl','wb') as f: pickle.dump(K_ven, f)
print('saved')

## Venue factors — method and findings

**Competition confound found first.** Initial venue factors put all nine CPL
grounds below 1.0 on runs (range 0.79-1.00, mean ~0.88). Nine out of nine is
not coincidence — it was the league-strength effect leaking into the venue
numbers. The pool is dominated by the IPL (87,690 balls, the only competition
scoring above the pool average), so every Caribbean ground inherited the gap
between CPL and IPL scoring.

**Fix:** compute a competition factor per phase first, then measure each
venue against what its own competition would expect. Venue factors now
scatter around 1.0 (five above, four below), isolating the pitch from the
league.

**Competition factors, CPL (runs):** powerplay 0.900, middle 0.887,
death 0.972. CPL scores well below pool average early and through the middle,
then nearly catches up at the death.

**CPL venue runs factors (middle overs, before shrinkage):**
Warner Park 1.126 (highest), Sabina Park 1.070, Tarouba 1.040,
Gros Islet 1.006, Kensington Oval 1.017, Providence 0.947,
North Sound 0.946, Arnos Vale 0.906, Queen's Park Oval 0.889 (lowest).

**Shrinkage K, out-of-sample:**

| Metric | Powerplay | Middle | Death |
|--------|-----------|--------|-------|
| Runs   | 4,000     | 4,000  | 2,000 |
| Wicket | none      | none   | none  |

**Venue wicket factors carry no predictive signal.** Extending the K search
to 10^9 (i.e. ignore the venue entirely) produced the lowest error in all
three phases, and the error curve was almost flat — 0.01839 at K=16,000 vs
0.01813 at K=10^9. Reason: wickets are rare (~1 per 20 balls), so Providence's
4,235 middle-over balls contain only ~210 wickets. Real venue differences are
smaller than the sampling noise. **Venue wicket factor set to 1.0 everywhere.**

Venue runs factors, by contrast, kept K=2,000-4,000 even with 10^9 available,
confirming genuine minima. Runs accumulate every ball and are measurable in a
way wicket rates are not.

**Practical effect of K=4,000:** even Providence (4,235 balls, best-sampled
CPL ground) gets only ~50% weight. Warner Park's 1.126 shrinks to ~1.07.
Sabina Park's 1.070 off 427 balls shrinks to ~1.01 — effectively neutral,
which is correct given Jamaica's return this season means very little recent
data for that ground.

**Note:** the original build plan had venue affecting both runs and wickets.
The data does not support the wicket half.

In [ ]:
def get_factor(df, name, ph, metric, role):
    """Return shrunk factor. metric: 'runs' or 'wicket'."""
    K = Ks[(role, metric, ph)]
    col = 'runs_factor' if metric == 'runs' else 'wkt_factor'
    try:
        row = df.loc[(name, ph)]
    except KeyError:
        return 1.0
    balls, raw = row['balls'], row[col]
    return (balls * raw + K) / (balls + K)


def venue_factor(v, ph):
    K = K_ven[('runs', ph)]
    try:
        row = ven.loc[(v, ph)]
    except KeyError:
        return 1.0
    return (row['balls'] * row['runs_factor'] + K) / (row['balls'] + K)

In [ ]:
for ph in ['powerplay','middle','death']:
    r = get_factor(bat, 'AD Russell', ph, 'runs', 'batter')
    w = get_factor(bat, 'AD Russell', ph, 'wicket', 'batter')
    print(f'{ph:<10} runs {r:.3f}  wkt {w:.3f}')

print()
print('Warner Park middle:', round(venue_factor('Warner Park, Basseterre, St Kitts','middle'), 3))
print('Providence middle:', round(venue_factor('Providence Stadium, Guyana','middle'), 3))

In [ ]:
print([n for n in bat.index.get_level_values('batter').unique() if 'Russell' in n])
print()
print(bat.xs('death', level='phase').loc['AD Russell'])

In [ ]:
def ball_probs(batter, bowler, ph, vfac):
    p = dist.loc[ph].values.astype(float).copy()

    br  = get_factor(bat,  batter, ph, 'runs',   'batter')
    bw  = get_factor(bat,  batter, ph, 'wicket', 'batter')
    or_ = get_factor(bowl, bowler, ph, 'runs',   'bowler')
    ow  = get_factor(bowl, bowler, ph, 'wicket', 'bowler')

    p_wkt = base.loc[ph,'wkt'] * bw * ow
    p_wkt = min(max(p_wkt, 0.005), 0.35)

    target = base.loc[ph,'rpb'] * br * or_ * vfac

    # find scale s on scoring shots that hits the target expected runs
    outs = outcomes.astype(float)
    for _ in range(40):
        s = target / max(sum(p[i]*outs[i] for i in range(7)) * (1-p_wkt), 1e-9)
        p[1:] = np.minimum(p[1:] * s, 1.0)
        tot = p[1:].sum()
        if tot > 0.98:
            p[1:] *= 0.98 / tot
            tot = 0.98
        p[0] = 1 - tot

    return p * (1 - p_wkt), p_wkt

In [ ]:
rng = np.random.default_rng(42)
res = [sim_ball('AD Russell','SP Narine','death',1.045,rng) for _ in range(10000)]
runs = sum(r for t,r in res if t=='R')
wkts = sum(1 for t,r in res if t=='W')
print(f'{runs/10000:.3f} runs per ball, {wkts/10000:.4f} wicket rate')

In [ ]:
def sim_innings(bat_order, bowl_attack, venue, rng, target=None):
    score, wkts, ball = 0, 0, 0
    strike, nonstrike, next_bat = 0, 1, 2

    for over in range(20):
        if wkts >= 10: break
        ph = phase(over)
        vfac = venue_factor(venue, ph)
        bowler = bowl_attack[over % len(bowl_attack)]

        for _ in range(6):
            if wkts >= 10: break
            if target is not None and score >= target: break

            kind, runs = sim_ball(bat_order[strike], bowler, ph, vfac, rng)
            ball += 1

            if kind == 'W':
                wkts += 1
                if next_bat < len(bat_order):
                    strike = next_bat
                    next_bat += 1
            else:
                score += runs
                if runs % 2 == 1:
                    strike, nonstrike = nonstrike, strike

        strike, nonstrike = nonstrike, strike   # end of over
        if target is not None and score >= target: break

    return score, wkts, ball

In [ ]:
known = sorted(set(bat.index.get_level_values('batter')) |
               set(bowl.index.get_level_values('bowler')))

def find(s):
    print(f'{s:<14} -> {[n for n in known if s.lower() in n.lower()]}')

for s in ['King','Carter','de Kock','Rutherford','Green','Clarke',
          'Anderson','Motie','Sams','Mujeeb','Simmonds',
          'Charles','Mayers','Shanaka','Holder','Fletcher','Hasaranga',
          'Athanaze','Bidaisee','Naseem','Louis','Salamkheil']:
    find(s)

In [ ]:
print([n for n in known if 'Wanindu' in n or 'Hasa' in n])

In [ ]:
print([n for n in known if 'Silva' in n])

In [ ]:
barbados = ['BA King','ZN Carter','Q de Kock','SE Rutherford','C Green',
            'RA Clarke','KA Anderson','G Motie','DR Sams',
            'Mujeeb Ur Rahman','RR Simmonds']

patriots = ['J Charles','KR Mayers','MD Shanaka','JO Holder','ADS Fletcher',
            'PWH de Silva','A Athanaze','N Bidaisee','Naseem Shah',
            'JS Louis','Waqar Salamkheil']

barb_attack = ['DR Sams','RR Simmonds','Mujeeb Ur Rahman','G Motie','SE Rutherford']
pat_attack  = ['Naseem Shah','JO Holder','PWH de Silva','JS Louis','KR Mayers']

# confirm everyone resolves
for n in barbados + patriots:
    if n not in known:
        print('MISSING:', n)
print('names checked')

In [ ]:
def phase(over):
    if over <= 5:   return 'powerplay'
    if over <= 14:  return 'middle'
    return 'death'

In [ ]:
rng = np.random.default_rng(7)
for _ in range(10):
    print(sim_innings(barbados, pat_attack, 'Warner Park, Basseterre, St Kitts', rng))

In [ ]:
rng = np.random.default_rng(7)
res = [sim_innings(barbados, pat_attack, 'Warner Park, Basseterre, St Kitts', rng)
       for _ in range(500)]
s = pd.Series([r[0] for r in res])
w = pd.Series([r[1] for r in res])
print(s.describe().round(1))
print()
print('mean wickets', round(w.mean(),2))
print('200+:', (s>=200).mean().round(3), ' 150-:', (s<150).mean().round(3))

In [ ]:
cpl = pool[pool['comp']=='cpl']
inn = cpl.groupby(['match','innings']).agg(
        runs=('runs_total','sum'), wkts=('wicket','sum'), balls=('legal','sum'))
full = inn[inn['balls']>=110]
print(full[['runs','wkts']].describe().round(1))
print('200+:', (full['runs']>=200).mean().round(3))

## Known limitation — under-dispersed innings totals

Validated the engine against 213 completed CPL innings (>=110 legal balls)
from the pool, simulating 500 Barbados Royals innings vs the SKN Patriots
attack at Warner Park.

| Metric        | Real CPL | Engine |
|---------------|----------|--------|
| Mean runs     | 166.4    | 169.9  |
| Mean wickets  | 6.6      | 6.56   |
| Std deviation | 27.7     | 23.6   |
| 200+ innings  | 11.7%    | 7.4%   |

**Centre is right, tails are too thin.** Mean runs and mean wickets both
match closely (the +3.5 runs is expected — Barbados are a strong batting
side and Warner Park is the highest-scoring CPL ground). But the engine
produces 200+ innings at 7.4% against a real 11.7%, and its standard
deviation is ~15% too low.

**Cause: ball-by-ball independence.** Each delivery is drawn independently
from a phase distribution adjusted by player and venue factors. Real innings
are not independent — a set batter goes on a four-over tear, or three
wickets fall in twelve balls and the innings folds. Those clusters generate
fat tails that independent draws average away.

**Consequence for pricing.** Match odds off this engine will be broadly
sensible (the centre is right) but will systematically underprice extreme
outcomes and overprice favourites in mismatches, because the model treats
close games as more certain than they are. Worst affected: innings-total
markets and any bet dependent on the tails.

**Fix (v2, not v1).** Partnership effects or intent states (defensive /
normal / push / full push), which let batters shift gear and generate the
acceleration and collapse patterns independence cannot.

Not fixed here deliberately — v1 is being validated before features are
added, one change at a time.

In [ ]:
def sim_match(teamA, attackA, teamB, attackB, venue, rng, a_bats_first=True):
    if a_bats_first:
        first, fatt, second, satt = teamA, attackB, teamB, attackA
    else:
        first, fatt, second, satt = teamB, attackA, teamA, attackB

    s1, w1, b1 = sim_innings(first, fatt, venue, rng)
    s2, w2, b2 = sim_innings(second, satt, venue, rng, target=s1+1)

    if s2 > s1:   winner = 'second'
    elif s1 > s2: winner = 'first'
    else:         winner = 'tie'
    return s1, w1, s2, w2, winner

In [ ]:
rng = np.random.default_rng(3)
for _ in range(5):
    print(sim_match(barbados, barb_attack, patriots, pat_attack,
                    'Warner Park, Basseterre, St Kitts', rng))

In [ ]:
import time

def price_match(teamA, attackA, teamB, attackB, venue, n=10000, seed=1):
    rng = np.random.default_rng(seed)
    a_wins = 0
    for i in range(n):
        a_first = (i % 2 == 0)          # alternate who bats first
        s1,w1,s2,w2,res = sim_match(teamA, attackA, teamB, attackB,
                                    venue, rng, a_bats_first=a_first)
        if res == 'tie':
            a_wins += 0.5
        elif (res == 'first') == a_first:
            a_wins += 1
    p = a_wins / n
    return p, 1/p, 1/(1-p)

t = time.time()
p, oddsA, oddsB = price_match(barbados, barb_attack, patriots, pat_attack,
                              'Warner Park, Basseterre, St Kitts', n=2000)
print(f'Barbados {p:.1%}  ({oddsA:.2f})   Patriots {1-p:.1%}  ({oddsB:.2f})')
print(f'{time.time()-t:.1f}s for 2000 sims')

In [ ]:
def build_lookup(team, attack, venue):
    L = {}
    for ph in ['powerplay','middle','death']:
        for p in team:
            L[('bat', p, ph)] = (get_factor(bat, p, ph, 'runs', 'batter'),
                                 get_factor(bat, p, ph, 'wicket', 'batter'))
        for p in attack:
            L[('bowl', p, ph)] = (get_factor(bowl, p, ph, 'runs', 'bowler'),
                                  get_factor(bowl, p, ph, 'wicket', 'bowler'))
        L[('ven', ph)] = venue_factor(venue, ph)
    return L


_pcache = {}

def ball_probs(batter, bowler, ph, L):
    key = (batter, bowler, ph)
    if key in _pcache:
        return _pcache[key]

    p = dist.loc[ph].values.astype(float).copy()
    br, bw = L[('bat', batter, ph)]
    or_, ow = L[('bowl', bowler, ph)]
    vfac = L[('ven', ph)]

    p_wkt = min(max(base.loc[ph,'wkt'] * bw * ow, 0.005), 0.35)
    target = base.loc[ph,'rpb'] * br * or_ * vfac

    outs = outcomes.astype(float)
    for _ in range(40):
        s = target / max((p*outs).sum() * (1-p_wkt), 1e-9)
        p[1:] = np.minimum(p[1:] * s, 1.0)
        tot = p[1:].sum()
        if tot > 0.98:
            p[1:] *= 0.98/tot
            tot = 0.98
        p[0] = 1 - tot

    res = (np.cumsum(p * (1-p_wkt)), p_wkt)
    _pcache[key] = res
    return res


def sim_innings(bat_order, bowl_attack, L, rng, target=None):
    score, wkts = 0, 0
    strike, nonstrike, next_bat = 0, 1, 2

    for over in range(20):
        if wkts >= 10: break
        ph = phase(over)
        bowler = bowl_attack[over % len(bowl_attack)]

        for _ in range(6):
            if wkts >= 10: break
            if target is not None and score >= target: break

            cum, p_wkt = ball_probs(bat_order[strike], bowler, ph, L)
            r = rng.random()
            if r < p_wkt:
                wkts += 1
                if next_bat < len(bat_order):
                    strike = next_bat; next_bat += 1
            else:
                runs = int(outcomes[np.searchsorted(cum, r - p_wkt)])
                score += runs
                if runs % 2 == 1:
                    strike, nonstrike = nonstrike, strike

        strike, nonstrike = nonstrike, strike
        if target is not None and score >= target: break

    return score, wkts

In [ ]:
def price_match(teamA, attackA, teamB, attackB, venue, n=10000, seed=1):
    L = build_lookup(teamA + teamB, attackA + attackB, venue)
    rng = np.random.default_rng(seed)

    a_wins = 0.0
    for i in range(n):
        a_first = (i % 2 == 0)
        if a_first:
            s1, _ = sim_innings(teamA, attackB, L, rng)
            s2, _ = sim_innings(teamB, attackA, L, rng, target=s1+1)
        else:
            s1, _ = sim_innings(teamB, attackA, L, rng)
            s2, _ = sim_innings(teamA, attackB, L, rng, target=s1+1)

        if s2 == s1:
            a_wins += 0.5
        elif (s2 > s1) != a_first:
            a_wins += 1
    p = a_wins / n
    return p, 1/p, 1/(1-p)

In [ ]:
import inspect
print('sim_match' in inspect.getsource(price_match))

In [ ]:
import time
_pcache.clear()
t = time.time()
p, oA, oB = price_match(barbados, barb_attack, patriots, pat_attack,
                        'Warner Park, Basseterre, St Kitts', n=10000)
print(f'Barbados {p:.1%} ({oA:.2f})   Patriots {1-p:.1%} ({oB:.2f})')
print(f'{time.time()-t:.1f}s for 10000 sims')

In [ ]:
for seed in [1,2,3,4,5]:
    p,_,_ = price_match(barbados, barb_attack, patriots, pat_attack,
                        'Warner Park, Basseterre, St Kitts', n=10000, seed=seed)
    print(seed, f'{p:.3%}')

In [ ]:
import pickle
with open('venue_factors.pkl','wb') as f: pickle.dump(ven, f)
with open('K_venue.pkl','wb') as f: pickle.dump(K_ven, f)
print('saved')

In [ ]:
import time
t = time.time()
for seed in [1,2,3]:
    p, oA, oB = price_match(barbados, barb_attack, patriots, pat_attack,
                            'Warner Park, Basseterre, St Kitts', n=50000, seed=seed)
    print(f'seed {seed}: {p:.3%}  ({oA:.3f} / {oB:.3f})')
print(f'{time.time()-t:.1f}s total')

In [ ]:
import os, glob, json, pickle, time
import pandas as pd, numpy as np
os.chdir('/Users/Jake/Documents/Cricket Comps Data')

pool = pd.read_pickle('pool.pkl')
base = pd.read_pickle('base.pkl')
dist = pd.read_pickle('dist.pkl')
bat  = pd.read_pickle('bat_factors.pkl')
bowl = pd.read_pickle('bowl_factors.pkl')
ven  = pd.read_pickle('venue_factors.pkl')
with open('shrinkage_K.pkl','rb') as f: Ks = pickle.load(f)
with open('K_venue.pkl','rb') as f: K_ven = pickle.load(f)

legal = pool[pool['legal']].copy()
outcomes = np.array([0,1,2,3,4,5,6])
_pcache = {}

print(len(pool), 'balls loaded')

In [4]:
p,_,_ = price_match(barbados, barb_attack, patriots, pat_attack,
                    'Warner Park, Basseterre, St Kitts', n=10000)
print(f'{p:.1%}')

55.7%


In [5]:
import json, glob

def match_setup(path):
    with open(path) as fh:
        m = json.load(fh)
    info = m['info']
    if 'venue' not in info or len(m.get('innings', [])) < 2:
        return None

    teams = info['teams']
    setup = {'date': info['dates'][0], 'venue': info['venue'],
             'teams': teams, 'outcome': info.get('outcome', {})}

    for i, inn in enumerate(m['innings'][:2]):
        order, seen = [], set()
        bowlers, bseen = [], set()
        for over in inn.get('overs', []):
            for d in over['deliveries']:
                for p in (d['batter'], d['non_striker']):
                    if p not in seen:
                        seen.add(p); order.append(p)
                if d['bowler'] not in bseen:
                    bseen.add(d['bowler']); bowlers.append(d['bowler'])
        setup[f'bat{i+1}'] = order
        setup[f'bowl{i+1}'] = bowlers
    return setup

paths = sorted(glob.glob('cpl_json/*.json'))
s = match_setup(paths[-1])
print(s['date'], s['venue'])
print(s['teams'])
print()
print('batting 1st:', s['bat1'][:6])
print('bowling vs them:', s['bowl1'])
print()
print(s['outcome'])

2016-07-31 Central Broward Regional Park Stadium Turf Ground
['St Lucia Zouks', 'Jamaica Tallawahs']

batting 1st: ['ADS Fletcher', 'J Charles', 'MEK Hussey', 'SR Watson', 'DJG Sammy']
bowling vs them: ['Imad Wasim', 'Shakib Al Hasan', 'KOK Williams', 'GE Mathurin', 'R Powell', 'TP Allen']

{'winner': 'St Lucia Zouks', 'by': {'runs': 17}}


In [6]:
setups = []
for p in paths:
    s = match_setup(p)
    if s and 'winner' in s.get('outcome', {}):
        setups.append(s)

setups.sort(key=lambda x: x['date'])
print(len(setups), 'usable matches')
print(setups[0]['date'], '->', setups[-1]['date'])

from collections import Counter
print()
print(Counter(s['venue'] for s in setups).most_common(12))

413 usable matches
2013-07-30 -> 2026-08-23

[('Warner Park, Basseterre, St Kitts', 53), ("Queen's Park Oval, Port of Spain", 45), ('Providence Stadium, Guyana', 40), ('Warner Park, Basseterre', 37), ('Brian Lara Stadium, Tarouba', 29), ('Providence Stadium', 28), ('Kensington Oval, Bridgetown', 26), ('Sabina Park, Kingston', 26), ('Daren Sammy National Cricket Stadium, Gros Islet', 25), ('Daren Sammy National Cricket Stadium, Gros Islet, St Lucia', 23), ('Brian Lara Stadium, Tarouba, Trinidad', 17), ('Kensington Oval, Bridgetown, Barbados', 15)]


In [7]:
def norm_venue(v):
    return ', '.join(v.split(', ')[:2]).strip()

from collections import Counter
print(Counter(norm_venue(s['venue']) for s in setups).most_common(15))

[('Warner Park, Basseterre', 90), ("Queen's Park Oval, Port of Spain", 54), ('Daren Sammy National Cricket Stadium, Gros Islet', 48), ('Brian Lara Stadium, Tarouba', 46), ('Kensington Oval, Bridgetown', 41), ('Providence Stadium, Guyana', 40), ('Sabina Park, Kingston', 30), ('Providence Stadium', 28), ('Sir Vivian Richards Stadium, North Sound', 18), ('Central Broward Regional Park Stadium Turf Ground', 13), ('Arnos Vale Ground, Kingstown', 3), ("National Cricket Stadium, St George's", 2)]


In [8]:
def norm_venue(v):
    return v.split(',')[0].strip()

print(Counter(norm_venue(s['venue']) for s in setups).most_common(15))

[('Warner Park', 90), ('Providence Stadium', 68), ("Queen's Park Oval", 54), ('Daren Sammy National Cricket Stadium', 48), ('Brian Lara Stadium', 46), ('Kensington Oval', 41), ('Sabina Park', 30), ('Sir Vivian Richards Stadium', 18), ('Central Broward Regional Park Stadium Turf Ground', 13), ('Arnos Vale Ground', 3), ('National Cricket Stadium', 2)]


In [9]:
legal2 = legal.copy()
legal2['venue'] = legal2['venue'].apply(norm_venue)

comp = (legal2.groupby(['comp','phase'])
        .apply(lambda g: pd.Series({
            'rpb': (g['runs_total']*g['w']).sum()/g['w'].sum(),
            'wpb': (g['wicket']*g['w']).sum()/g['w'].sum()})))
comp = comp.join(base[['rpb','wkt']].rename(columns={'rpb':'b_r','wkt':'b_w'}), on='phase')
comp['comp_runs'] = comp['rpb']/comp['b_r']
comp['comp_wkt']  = comp['wpb']/comp['b_w']

legal2 = legal2.join(comp[['comp_runs','comp_wkt']], on=['comp','phase'])

ven = (legal2.groupby(['venue','phase'])
       .apply(lambda g: pd.Series({
           'balls': g['legal'].sum(),
           'exp_r': (g['comp_runs']*g['w']).sum(),
           'act_r': (g['runs_total']*g['w']).sum()})))
ven = ven.join(base[['rpb']].rename(columns={'rpb':'b_r'}), on='phase')
ven['runs_factor'] = ven['act_r'] / (ven['exp_r'] * ven['b_r'])

_pcache.clear()
print(ven.xs('middle', level='phase').loc[
    ['Warner Park','Providence Stadium',"Queen's Park Oval"]][['balls','runs_factor']].round(3))

                     balls  runs_factor
venue                                  
Warner Park         2231.0        1.126
Providence Stadium  4235.0        0.947
Queen's Park Oval    849.0        0.889


In [10]:
print([v for v in legal['venue'].unique() if 'Providence' in v])
print([v for v in legal['venue'].unique() if 'Warner' in v])

['Providence Stadium, Guyana']
['Warner Park, Basseterre, St Kitts']


In [11]:
canon = {}
for v in legal['venue'].unique():
    canon[norm_venue(v)] = v

# check the CPL grounds all resolve
for s_v in ['Warner Park','Providence Stadium',"Queen's Park Oval",
            'Daren Sammy National Cricket Stadium','Brian Lara Stadium',
            'Kensington Oval','Sabina Park','Sir Vivian Richards Stadium',
            'Arnos Vale Ground','National Cricket Stadium',
            'Central Broward Regional Park Stadium Turf Ground']:
    print(f'{s_v:<50} -> {canon.get(s_v, "NOT IN POOL")}')

Warner Park                                        -> Warner Park, Basseterre, St Kitts
Providence Stadium                                 -> Providence Stadium, Guyana
Queen's Park Oval                                  -> Queen's Park Oval, Port of Spain, Trinidad
Daren Sammy National Cricket Stadium               -> Daren Sammy National Cricket Stadium, Gros Islet, St Lucia
Brian Lara Stadium                                 -> Brian Lara Stadium, Tarouba, Trinidad
Kensington Oval                                    -> Kensington Oval, Bridgetown, Barbados
Sabina Park                                        -> Sabina Park, Kingston, Jamaica
Sir Vivian Richards Stadium                        -> Sir Vivian Richards Stadium, North Sound, Antigua
Arnos Vale Ground                                  -> Arnos Vale Ground, Kingstown, St Vincent
National Cricket Stadium                           -> NOT IN POOL
Central Broward Regional Park Stadium Turf Ground  -> Central Broward Regional Park Sta

In [12]:
n25 = sum(1 for s in setups if s['date'] >= '2025-01-01')
print(n25, 'matches from 2025 onward')
n24 = sum(1 for s in setups if s['date'] >= '2024-01-01')
print(n24, 'from 2024 onward')

47 matches from 2025 onward
81 from 2024 onward


In [13]:
def build_factors(cutoff_date):
    """Factors from all pool data strictly before cutoff_date."""
    L = legal[legal['date'] < cutoff_date].copy()
    latest = L['date'].max()
    L['w'] = 0.5 ** (((latest - L['date']).dt.days/365.25)/2.0)

    b = base   # phase baselines held fixed
    def mk(df, key, runcol):
        g = (df.groupby([key,'phase'])
               .apply(lambda x: pd.Series({
                   'balls': x['legal'].sum(),
                   'eff': x['w'].sum(),
                   'r': (x[runcol]*x['w']).sum(),
                   'k': (x['wicket']*x['w']).sum()})))
        g = g.join(b[['rpb','wkt']].rename(columns={'rpb':'br','wkt':'bw'}), on='phase')
        g['runs_factor'] = (g['r']/g['eff'])/g['br']
        g['wkt_factor']  = (g['k']/g['eff'])/g['bw']
        return g

    return mk(L,'batter','runs_bat'), mk(L,'bowler','runs_total')

bat23, bowl23 = build_factors('2023-01-01')
print(len(bat23), 'batter-phase rows for the 2023 backtest')

1360 batter-phase rows for the 2023 backtest


In [14]:
seasons = ['2023','2024','2025','2026']
factor_sets = {}
for yr in seasons:
    factor_sets[yr] = build_factors(f'{yr}-01-01')
    print(yr, 'factors built')

2023 factors built
2024 factors built
2025 factors built
2026 factors built


In [18]:
def match_setup(path):
    with open(path) as fh:
        m = json.load(fh)
    info = m['info']
    if 'venue' not in info or len(m.get('innings', [])) < 2:
        return None

    setup = {'date': info['dates'][0], 'venue': info['venue'],
             'teams': info['teams'], 'outcome': info.get('outcome', {})}

    for i, inn in enumerate(m['innings'][:2]):
        setup[f'team{i+1}'] = inn['team']
        order, seen = [], set()
        bowlers, bseen = [], set()
        for over in inn.get('overs', []):
            for d in over['deliveries']:
                for p in (d['batter'], d['non_striker']):
                    if p not in seen:
                        seen.add(p); order.append(p)
                if d['bowler'] not in bseen:
                    bseen.add(d['bowler']); bowlers.append(d['bowler'])
        setup[f'bat{i+1}'] = order
        setup[f'bowl{i+1}'] = bowlers
    return setup


setups = []
for p in paths:
    s = match_setup(p)
    if s and 'winner' in s.get('outcome', {}):
        setups.append(s)
setups.sort(key=lambda x: x['date'])

print(len(setups), 'matches')
s = setups[-1]
print(s['date'], '|', s['team1'], 'batted first |', s['team2'], 'chased')
print('winner:', s['outcome']['winner'])

413 matches
2026-08-23 | Antigua and Barbuda Falcons batted first | Guyana Amazon Warriors chased
winner: Guyana Amazon Warriors


In [19]:
import time

results = []
t = time.time()

for s in setups:
    yr = s['date'][:4]
    if yr not in factor_sets:
        continue

    bat, bowl = factor_sets[yr]
    _pcache.clear()

    venue = canon.get(norm_venue(s['venue']), s['venue'])

    # team1 batted first, and bowl2 is the attack that bowled at them
    p_first, _, _ = price_match(s['bat1'], s['bowl2'],
                                s['bat2'], s['bowl1'],
                                venue, n=2000, seed=42)

    results.append({
        'date':    s['date'],
        'venue':   norm_venue(s['venue']),
        'first':   s['team1'],
        'second':  s['team2'],
        'p_first': p_first,
        'won_first': int(s['outcome']['winner'] == s['team1']),
    })

bt = pd.DataFrame(results)
print(len(bt), 'matches priced in', round(time.time()-t), 's')
print()
print('model mean P(bat first wins):', round(bt['p_first'].mean(), 3))
print('actual bat-first win rate:   ', round(bt['won_first'].mean(), 3))

111 matches priced in 67 s

model mean P(bat first wins): 0.493
actual bat-first win rate:    0.414


In [20]:
from collections import Counter
print(Counter(s['date'][:4] for s in setups))
print(list(factor_sets.keys()))

Counter({'2024': 34, '2017': 33, '2018': 33, '2019': 32, '2020': 32, '2021': 32, '2025': 32, '2015': 31, '2022': 31, '2023': 30, '2016': 28, '2014': 27, '2013': 23, '2026': 15})
['2023', '2024', '2025', '2026']


In [21]:
print(bt.groupby(bt['date'].str[:4]).agg(
    n=('won_first','size'),
    model=('p_first','mean'),
    actual=('won_first','mean')).round(3))

print()
print('Brier:', round(((bt['p_first']-bt['won_first'])**2).mean(), 4))
print('Brier if always 0.5:', 0.25)

       n  model  actual
date                   
2023  30  0.480   0.500
2024  34  0.502   0.412
2025  32  0.494   0.406
2026  15  0.498   0.267

Brier: 0.2325
Brier if always 0.5: 0.25


In [22]:
recent = bt[bt['date'] >= '2024-01-01']
print(len(recent), 'matches 2024+')
print('model', round(recent['p_first'].mean(),3),
      'actual', round(recent['won_first'].mean(),3))

import math
n = len(recent)
se = math.sqrt(0.25/n)
diff = recent['p_first'].mean() - recent['won_first'].mean()
print(f'gap {diff:.3f}, se {se:.3f}, z = {diff/se:.2f}')

81 matches 2024+
model 0.498 actual 0.383
gap 0.115, se 0.056, z = 2.07


## Backtest — first real test of the model

**What was tested.** The model priced 111 CPL matches from 2023 to 2026
that it had never seen. For each match it was given the two teams, the
ground, and nothing else — then it simulated the match 2,000 times and
counted how often each side won. That count is its prediction.

**Keeping it honest.** A model must not be tested on matches it learned
from, or it is marking its own homework. So player ratings were rebuilt
four times, once per season, each using only cricket played *before* that
season started. The model predicting a 2025 match knows nothing about 2025.

**The headline result.** Accuracy is measured with a Brier score, where
lower is better and a coin-flip scores 0.250. The model scored **0.2325**.
It beats guessing. That is a modest edge rather than a large one, and it
has not yet been compared against bookmakers' prices, which is the harder
test.

**A possible blind spot.** In cricket one side bats first and the other
chases the total. Across the 111 matches the model expected the team
batting first to win 49% of the time. They actually won 41%. Chasing sides
did better than the model thought.

That gap could be real: chasing is often easier because you know exactly
what you need, and evening dew makes the ball harder to bowl with. The
model cannot see either of these — it simulates the second innings the
same way as the first.

But it could also be chance. Across 111 matches a gap this size happens by
luck roughly one time in eleven. Splitting the data to 2024 onwards makes
it look stronger, but that split was chosen after seeing which years looked
bad, which is not a fair test.

**Decision: do not fix it yet.** Rather than adding a correction fitted to
the same 111 matches that suggested it, the model will price the remaining
CPL fixtures of this season unchanged, and those results will be checked
against the same question. If chasing sides keep outperforming, that is
independent evidence and the fix is justified. If they do not, a correction
fitted to noise was avoided.

**Caveat.** Ground and phase ratings were held fixed rather than rebuilt
each season, unlike the player ratings. This is a small amount of leakage
and should be tidied in a later version.

In [23]:
known = sorted(set(bat.index.get_level_values('batter')) |
               set(bowl.index.get_level_values('bowler')))

def find(s):
    print(f'{s:<14} -> {[n for n in known if s.lower() in n.lower()]}')

for s in ['Munro','Narine','Pooran','Hales','Goolie','Greaves','Hosein',
          'Drakes','Da Silva','Kumara','Tariq',
          'Cornwall','Jangoo','Gore','Nawaz','Moeen','Shadab','Springer',
          'James','Joseph','Muqeem','Moqim','Seales']:
    find(s)

Munro          -> ['C Munro']
Narine         -> ['SP Narine']
Pooran         -> ['N Pooran']
Hales          -> ['AD Hales']
Goolie         -> ['JU Goolie']
Greaves        -> ['JP Greaves']
Hosein         -> ['AJ Hosein']
Drakes         -> ['DC Drakes']
Da Silva       -> ['J Da Silva']
Kumara         -> ['CBRLS Kumara']
Tariq          -> ['Usman Tariq']
Cornwall       -> ['RRS Cornwall']
Jangoo         -> ['AA Jangoo']
Gore           -> ['K Gore']
Nawaz          -> ['Fahad Nawaz', 'Hassan Nawaz']
Moeen          -> []
Shadab         -> ['Shadab Khan']
Springer       -> ['SK Springer']
James          -> ['JM James', 'James Bracey', 'KHM James', 'LW James']
Joseph         -> ['AS Joseph', 'S Joseph']
Muqeem         -> []
Moqim          -> []
Seales         -> ['JNT Seales']


In [24]:
print([n for n in known if 'Sufyan' in n])
print([n for n in known if 'ufyan' in n or 'oqim' in n or 'uqeem' in n])
print([n for n in known if 'Ali' in n])

[]
[]
['Ali Khan', 'Ali Naseer', 'Ali Sheikh', 'Asif Ali', 'Haider Ali', 'Hasan Ali', 'Kashif Ali', 'MM Ali', 'Sabir Ali']


In [25]:
tkr = ['C Munro','SP Narine','N Pooran','AD Hales','JU Goolie','JP Greaves',
       'AJ Hosein','DC Drakes','J Da Silva','CBRLS Kumara','Usman Tariq']

abf = ['RRS Cornwall','AA Jangoo','K Gore','Hassan Nawaz','MM Ali','Shadab Khan',
       'SK Springer','JM James','AS Joseph','Sufyan Moqim','JNT Seales']

tkr_attack = ['AJ Hosein','SP Narine','CBRLS Kumara','Usman Tariq','JU Goolie','DC Drakes']
abf_attack = ['JNT Seales','AS Joseph','RRS Cornwall','SK Springer','JM James','Shadab Khan']

for n in tkr + abf:
    if n not in known:
        print('fallback to average:', n)

_pcache.clear()
p, oT, oA = price_match(tkr, tkr_attack, abf, abf_attack,
                        "Queen's Park Oval, Port of Spain, Trinidad",
                        n=10000, seed=1)
print(f'\nTKR {p:.1%} ({oT:.2f})   ABF {1-p:.1%} ({oA:.2f})')

fallback to average: Sufyan Moqim

TKR 63.7% (1.57)   ABF 36.3% (2.75)


In [26]:
from datetime import datetime
import pandas as pd, os

row = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'date': '2026-09-02', 'comp': 'CPL',
    'home': 'Trinbago Knight Riders', 'away': 'Antigua & Barbuda Falcons',
    'venue': "Queen's Park Oval",
    'xi_status': 'expected',
    'judgment_p': 0.65,
    'model_p': round(p, 4),
    'model_odds': round(oT, 2),
    'caveats': 'Sufyan Moqim not in pool - treated as average',
    'bf_at_prediction': None, 'bf_at_close': None, 'result': None,
}

f = 'cpl_predictions.csv'
df = pd.DataFrame([row])
df.to_csv(f, mode='a', header=not os.path.exists(f), index=False)
print('logged')
print(df[['timestamp','model_p','judgment_p']].to_string(index=False))

logged
          timestamp  model_p  judgment_p
2026-09-02T22:04:35    0.637        0.65


In [27]:
df = pd.read_csv('cpl_predictions.csv')
df.loc[df.index[-1], 'bf_at_prediction'] = 1.66
df.to_csv('cpl_predictions.csv', index=False)
print(df.tail(1).to_string(index=False))

          timestamp       date comp                   home                      away             venue xi_status  judgment_p  model_p  model_odds                                       caveats  bf_at_prediction  bf_at_close  result
2026-09-02T22:04:35 2026-09-02  CPL Trinbago Knight Riders Antigua & Barbuda Falcons Queen's Park Oval  expected        0.65    0.637        1.57 Sufyan Moqim not in pool - treated as average              1.66          NaN     NaN


In [4]:
df = pd.read_csv('cpl_predictions.csv')

df.loc[df.index[-1], 'bf_pre_toss'] = 1.56
df.loc[df.index[-1], 'bf_at_close'] = 1.54
df.loc[df.index[-1], 'result'] = 0        # 0 = TKR lost, 1 = would be a win
df.loc[df.index[-1], 'toss_winner'] = 'TKR'
df.loc[df.index[-1], 'toss_decision'] = 'bowl'
df.loc[df.index[-1], 'caveats'] = ('Sufyan Moqim not in pool; priced off expected XIs '
                                   '- actual XIs differed (Pollard, Hinds in for TKR; '
                                   'Lewis, Allen in, Alzarri out for ABF)')

df.to_csv('cpl_predictions.csv', index=False)
print(df.tail(1).T)

                                                                  0
timestamp                                       2026-09-02T22:04:35
date                                                     2026-09-02
comp                                                            CPL
home                                         Trinbago Knight Riders
away                                      Antigua & Barbuda Falcons
venue                                             Queen's Park Oval
xi_status                                                  expected
judgment_p                                                     0.65
model_p                                                       0.637
model_odds                                                     1.57
caveats           Sufyan Moqim not in pool; priced off expected ...
bf_at_prediction                                               1.66
bf_at_close                                                    1.54
result                                          

In [9]:
known = sorted(set(bat.index.get_level_values('batter')) |
               set(bowl.index.get_level_values('bowler')))

def find(s):
    print(f'{s:<14} -> {[n for n in known if s.lower() in n.lower()]}')

for s in ['Gurbaz','Dindyal','Hope','Hetmyer','Nabi','Sampson','Shepherd',
          'Pretorius','Pierre','Shamar Joseph','Tahir']:
    find(s)

Gurbaz         -> ['Rahmanullah Gurbaz']
Dindyal        -> ['M Dindyal']
Hope           -> ['SD Hope']
Hetmyer        -> ['SO Hetmyer']
Nabi           -> ['Auqib Nabi', 'Mohammad Nabi']
Sampson        -> ['Q Sampson']
Shepherd       -> ['R Shepherd']
Pretorius      -> ['D Pretorius', 'LG Pretorius', 'M Pretorius']
Pierre         -> ['K Pierre']
Shamar Joseph  -> []
Tahir          -> ['Imran Tahir']


In [10]:
print([n for n in known if 'Joseph' in n])

['AS Joseph', 'S Joseph']


In [11]:
guy = ['Rahmanullah Gurbaz','M Dindyal','SD Hope','SO Hetmyer','Mohammad Nabi',
       'Q Sampson','R Shepherd','D Pretorius','K Pierre','S Joseph','Imran Tahir']

abf = ['E Lewis','RRS Cornwall','AA Jangoo','Hassan Nawaz','MM Ali','Shadab Khan',
       'FA Allen','SK Springer','JM James','Sufyan Moqim','JNT Seales']

guy_attack = ['D Pretorius','K Pierre','R Shepherd','S Joseph','Imran Tahir','Mohammad Nabi']
abf_attack = ['JNT Seales','SK Springer','JM James','Shadab Khan','Sufyan Moqim','RRS Cornwall']

for n in guy + abf:
    if n not in known:
        print('fallback to average:', n)

fallback to average: Sufyan Moqim


In [12]:
_pcache.clear()
p, oG, oA = price_match(guy, guy_attack, abf, abf_attack,
                        'Providence Stadium, Guyana', n=10000, seed=1)
print(f'Guyana {p:.1%} ({oG:.2f})   ABF {1-p:.1%} ({oA:.2f})')

Guyana 67.6% (1.48)   ABF 32.4% (3.08)


In [13]:
df = pd.read_csv('cpl_predictions.csv')
df.loc[df.index[-1], 'model_p'] = round(p, 4)
df.loc[df.index[-1], 'model_odds'] = round(oG, 2)
df.loc[df.index[-1], 'row_type'] = 'pre_toss_expected'
df.loc[df.index[-1], 'caveats'] = 'Sufyan Moqim not in pool - treated as average'
df.to_csv('cpl_predictions.csv', index=False)
print(df.tail(1).T)

                                                              2
timestamp                                   2026-09-08T00:00:00
date                                                 2026-09-08
comp                                                        CPL
home                                     Guyana Amazon Warriors
away                                  Antigua & Barbuda Falcons
venue                                        Providence Stadium
xi_status                                              expected
judgment_p                                                 0.58
model_p                                                  0.6755
model_odds                                                 1.48
caveats           Sufyan Moqim not in pool - treated as average
bf_at_prediction                                           1.57
bf_at_close                                                 NaN
result                                                      NaN
bf_pre_toss                             

In [14]:
def price_match_toss(bat_first, attack_first, bat_second, attack_second,
                     venue, n=10000, seed=1):
    """bat_first bats first; returns P(bat_first wins)."""
    L = build_lookup(bat_first + bat_second, attack_first + attack_second, venue)
    rng = np.random.default_rng(seed)
    wins = 0.0
    for _ in range(n):
        s1, _ = sim_innings(bat_first,  attack_second, L, rng)
        s2, _ = sim_innings(bat_second, attack_first,  L, rng, target=s1+1)
        if s2 == s1: wins += 0.5
        elif s1 > s2: wins += 1
    p = wins / n
    return p, 1/p, 1/(1-p)


abf2 = ['E Lewis','AA Jangoo','K Gore','Hassan Nawaz','MM Ali','Shadab Khan',
        'FA Allen','JM James','AS Joseph','Sufyan Moqim','JNT Seales']
abf2_attack = ['JNT Seales','AS Joseph','JM James','Shadab Khan','Sufyan Moqim','FA Allen']

for n_ in abf2:
    if n_ not in known:
        print('fallback:', n_)

_pcache.clear()
# ABF bat first, Guyana chase
p_abf, o_abf, o_guy = price_match_toss(abf2, abf2_attack, guy, guy_attack,
                                       'Providence Stadium, Guyana', n=10000, seed=1)
print(f'\nABF {p_abf:.1%} ({o_abf:.2f})   Guyana {1-p_abf:.1%} ({o_guy:.2f})')

fallback: Sufyan Moqim

ABF 32.6% (3.06)   Guyana 67.3% (1.48)


In [18]:
df = pd.read_csv('cpl_predictions.csv')
df = df.drop(index=1).reset_index(drop=True)

df.loc[0, 'row_type'] = 'pre_toss_expected'
df.loc[1, 'bf_at_prediction'] = 1.57
df.loc[1, 'bf_pre_toss'] = 1.60

df.to_csv('cpl_predictions.csv', index=False)
print(df[['date','row_type','judgment_p','model_p',
          'bf_at_prediction','bf_pre_toss','bf_at_close','result']].to_string(index=False))

      date            row_type  judgment_p  model_p  bf_at_prediction  bf_pre_toss  bf_at_close  result
2026-09-02   pre_toss_expected        0.65   0.6370              1.66         1.56         1.54     0.0
2026-09-08   pre_toss_expected        0.58   0.6755              1.57         1.60          NaN     NaN
2026-09-08 post_toss_confirmed        0.58   0.6735               NaN          NaN         1.46     NaN


In [19]:
df['judgment_odds'] = (1/df['judgment_p']).round(2)
df['model_odds'] = (1/df['model_p']).round(2)
df.to_csv('cpl_predictions.csv', index=False)

In [22]:
df = pd.read_csv('cpl_predictions.csv')

df['judgment_odds'] = (1 / df['judgment_p']).round(2)
df['model_odds']    = (1 / df['model_p']).round(2)

df.to_csv('cpl_predictions.csv', index=False)
print(df[['date','row_type','judgment_p','judgment_odds',
          'model_p','model_odds','bf_at_prediction','bf_pre_toss','bf_at_close']]
      .to_string(index=False))

      date            row_type  judgment_p  judgment_odds  model_p  model_odds  bf_at_prediction  bf_pre_toss  bf_at_close
2026-09-02   pre_toss_expected        0.65           1.54   0.6370        1.57              1.66         1.56         1.54
2026-09-08 post_toss_confirmed        0.58           1.72   0.6735        1.48              1.57         1.60         1.46
2026-09-08 post_toss_confirmed        0.58           1.72   0.6735        1.48               NaN          NaN         1.46


In [24]:
df = pd.read_csv('cpl_predictions.csv')
df.loc[2, 'judgment_p'] = 0.63
df.loc[2, 'judgment_odds'] = round(1/0.63, 2)
df.to_csv('cpl_predictions.csv', index=False)
print(df[['date','row_type','judgment_odds','model_odds',
          'bf_pre_toss','bf_at_close']].to_string(index=False))

      date            row_type  judgment_odds  model_odds  bf_pre_toss  bf_at_close
2026-09-02   pre_toss_expected           1.54        1.57         1.56         1.54
2026-09-08   pre_toss_expected           1.72        1.48         1.60          NaN
2026-09-08 post_toss_confirmed           1.59        1.48          NaN         1.46


In [5]:
df = pd.read_csv('cpl_predictions.csv')

df['model_err']    = (df['model_p']    - df['result']).abs()
df['judgment_err'] = (df['judgment_p'] - df['result']).abs()
df['closer'] = df.apply(
    lambda r: 'judgment' if r['judgment_err'] < r['model_err'] else 'model', axis=1)

df.to_csv('cpl_predictions.csv', index=False)
print(df[['date','row_type','judgment_p','model_p','result',
          'judgment_err','model_err','closer']].to_string(index=False))

      date            row_type  judgment_p  model_p  result  judgment_err  model_err   closer
2026-09-02   pre_toss_expected        0.65   0.6370     0.0          0.65     0.6370    model
2026-09-08   pre_toss_expected        0.58   0.6755     0.0          0.58     0.6755 judgment
2026-09-08 post_toss_confirmed        0.63   0.6735     0.0          0.63     0.6735 judgment
